## Kaggle Competition Project Info
Link <https://www.kaggle.com/competitions/WiDSWorldWide_GlobalDathon26> 

### Overview
When a wildfire ignites, emergency managers face an impossible question with incomplete information. Which fires will reach populated areas? How quickly? And which communities should prepare first?

This year's WiDS Global Datathon challenges participants to build survival models that answer these questions using only the earliest signals available. Your task is to predict the probability that a wildfire will threaten an evacuation zone within 12, 24, 48, and 72 hours, drawing on data from just the first five hours after ignition.

The goal is not a single prediction but a calibrated forecast across multiple time horizons. Emergency responders need both urgency rankings (which fires demand immediate attention) and probability estimates they can trust when making high-stakes decisions about evacuations, resource deployment, and public alerts.

In [53]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)


import os,json, time
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, train_test_split
import lightgbm as lgb

from sklearn.metrics import make_scorer
from sklearn.preprocessing import StandardScaler
from scipy.stats import norm, lognorm, expon  # for parametric survival curve approximation

import xgboost as xgb
from sklearn.metrics import make_scorer  # if you want custom metrics later
from sklearn.model_selection import StratifiedKFold
import numpy as np

In [54]:
class CFG:
    MetaDataFile = "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/metaData.csv"
    sampleFile = "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/sample_submission.csv"
    testFile = "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/test.csv"
    trainFile = "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/train.csv"

    TIME_COL = "time_to_hit_hours"
    EVENT_COL = "event"           # 1 = observed (hit), 0 = censored

In [55]:
metaData= pd.read_csv(CFG.MetaDataFile)
metaData

,column,type,category,description,units,range
0,event_id,identifier,identifier,Anonymized fire event identifier (stable rando...,NaN,NaN
1,time_to_hit_hours,target,target,Time from t0+5h until fire comes within 5km of...,hours,"[0, 72]"
2,event,target,target,"Event indicator: 1 if fire hit within 72h, 0 i...",NaN,NaN
3,num_perimeters_0_5h,feature,temporal_coverage,Number of perimeters within first 5 hours,NaN,NaN
4,dt_first_last_0_5h,feature,temporal_coverage,Time span between first and last perimeter (ho...,NaN,NaN
5,low_temporal_resolution_0_5h,feature,temporal_coverage,"Flag: 1 if dt < 0.5h or only 1 perimeter, else 0",NaN,NaN
6,area_first_ha,feature,growth,Initial fire area at t0 (hectares),NaN,NaN
7,area_growth_abs_0_5h,feature,growth,Feature from growth category,NaN,NaN
8,area_growth_rel_0_5h,feature,growth,Feature from growth category,NaN,NaN
9,area_growth_rate_ha_per_h,feature,growth,Area growth rate (hectares per hour),NaN,NaN


In [56]:
trainDF = pd.read_csv(CFG.trainFile)
trainDF

,event_id,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h,area_first_ha,area_growth_abs_0_5h,area_growth_rel_0_5h,area_growth_rate_ha_per_h,log1p_area_first,log1p_growth,...,dist_fit_r2_0_5h,alignment_cos,alignment_abs,cross_track_component,along_track_speed,event_start_hour,event_start_dayofweek,event_start_month,time_to_hit_hours,event
0,10892457,3,4.265188,0,79.696304,2.875935,0.036086,0.674281,4.390693,1.354787,...,0.886373,-0.054649,0.054649,-1.937219,-0.106026,19,4,5,18.892512,0
1,11757157,2,1.169918,0,8.946749,0.000000,0.000000,0.000000,2.297246,0.000000,...,0.000000,-0.568898,0.568898,-0.000000,-0.000000,4,4,6,22.048108,1
2,11945086,4,4.777526,0,106.482638,0.000000,0.000000,0.000000,4.677329,0.000000,...,0.000000,0.882385,0.882385,0.000000,0.000000,22,4,8,0.888895,1
3,12044083,1,0.000000,1,67.631125,0.000000,0.000000,0.000000,4.228746,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,20,5,8,60.953021,0
4,12052347,2,4.975273,0,35.632874,0.000000,0.000000,0.000000,3.600946,0.000000,...,0.000000,0.934634,0.934634,-0.000000,0.000000,21,5,7,44.990274,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
216,97075632,1,0.000000,1,51.295195,0.000000,0.000000,0.000000,3.956904,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,17,6,6,66.340624,0
217,97362560,2,1.127102,0,1.176991,0.000000,0.000000,0.000000,0.777943,0.000000,...,0.000000,-0.277779,0.277779,-0.000000,-0.000000,18,1,7,5.694898,1
218,97805715,2,3.710653,0,71.946930,0.000000,0.000000,0.000000,4.289732,0.000000,...,0.000000,0.694609,0.694609,0.000000,0.000000,18,1,9,44.011253,0
219,99071478,1,0.000000,1,20.223659,0.000000,0.000000,0.000000,3.055117,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,15,0,8,22.975783,1


In [5]:
trainDF.columns

Index(['event_id', 'num_perimeters_0_5h', 'dt_first_last_0_5h',
       'low_temporal_resolution_0_5h', 'area_first_ha', 'area_growth_abs_0_5h',
       'area_growth_rel_0_5h', 'area_growth_rate_ha_per_h', 'log1p_area_first',
       'log1p_growth', 'log_area_ratio_0_5h', 'relative_growth_0_5h',
       'radial_growth_m', 'radial_growth_rate_m_per_h',
       'centroid_displacement_m', 'centroid_speed_m_per_h',
       'spread_bearing_deg', 'spread_bearing_sin', 'spread_bearing_cos',
       'dist_min_ci_0_5h', 'dist_std_ci_0_5h', 'dist_change_ci_0_5h',
       'dist_slope_ci_0_5h', 'closing_speed_m_per_h',
       'closing_speed_abs_m_per_h', 'projected_advance_m',
       'dist_accel_m_per_h2', 'dist_fit_r2_0_5h', 'alignment_cos',
       'alignment_abs', 'cross_track_component', 'along_track_speed',
       'event_start_hour', 'event_start_dayofweek', 'event_start_month',
       'time_to_hit_hours', 'event'],
      dtype='object')

In [6]:
testDF = pd.read_csv(CFG.testFile)
testDF

,event_id,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h,area_first_ha,area_growth_abs_0_5h,area_growth_rel_0_5h,area_growth_rate_ha_per_h,log1p_area_first,log1p_growth,...,projected_advance_m,dist_accel_m_per_h2,dist_fit_r2_0_5h,alignment_cos,alignment_abs,cross_track_component,along_track_speed,event_start_hour,event_start_dayofweek,event_start_month
0,10662602,1,0.000000,1,2.452217,0.000000,0.00000,0.000000,1.239017,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,3,7
1,13353600,1,0.000000,1,131.669588,0.000000,0.00000,0.000000,4.887862,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,22,0,8
2,13942327,1,0.000000,1,6.723104,0.000000,0.00000,0.000000,2.044216,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2,6,7
3,16112781,1,0.000000,1,285.416736,0.000000,0.00000,0.000000,5.657448,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,1,7
4,17132808,7,3.459331,0,61.098604,12.516633,0.20486,3.618224,4.128724,2.603921,...,13.54413,-22.687575,0.044572,0.158550,0.158550,-24.414806,3.920562,23,5,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,94627327,2,4.614923,0,24.438326,0.000000,0.00000,0.000000,3.236257,0.000000,...,0.00000,0.000000,0.000000,0.763809,0.763809,0.000000,0.000000,22,0,6
91,96570675,1,0.000000,1,155.843418,0.000000,0.00000,0.000000,5.055248,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,21,2,7
92,97225766,1,0.000000,1,87.761553,0.000000,0.00000,0.000000,4.485954,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4,1,7
93,98446281,2,4.335964,0,11.391108,0.000000,0.00000,0.000000,2.516979,0.000000,...,0.00000,0.000000,0.000000,-0.305683,0.305683,0.000000,-0.000000,17,0,9


In [66]:
testDF["event_id"]

event_id
10662602    1
13353600    1
13942327    1
16112781    1
17132808    1
           ..
94627327    1
96570675    1
97225766    1
98446281    1
99649465    1
Name: count, Length: 95, dtype: int64

In [7]:
sample = pd.read_csv(CFG.sampleFile)
sample

,event_id,prob_12h,prob_24h,prob_48h,prob_72h
0,10662602,0.5,0.5,0.5,0.5
1,13353600,0.5,0.5,0.5,0.5
2,13942327,0.5,0.5,0.5,0.5
3,16112781,0.5,0.5,0.5,0.5
4,17132808,0.5,0.5,0.5,0.5
...,...,...,...,...,...
90,94627327,0.5,0.5,0.5,0.5
91,96570675,0.5,0.5,0.5,0.5
92,97225766,0.5,0.5,0.5,0.5
93,98446281,0.5,0.5,0.5,0.5


In [8]:
## EDA

In [9]:
# extract Features  — drop id, time, event
features = [col for col in trainDF.columns 
            if col not in [CFG.TIME_COL, CFG.EVENT_COL, "id"]]   # adjust "id" if present
features

['event_id',
 'num_perimeters_0_5h',
 'dt_first_last_0_5h',
 'low_temporal_resolution_0_5h',
 'area_first_ha',
 'area_growth_abs_0_5h',
 'area_growth_rel_0_5h',
 'area_growth_rate_ha_per_h',
 'log1p_area_first',
 'log1p_growth',
 'log_area_ratio_0_5h',
 'relative_growth_0_5h',
 'radial_growth_m',
 'radial_growth_rate_m_per_h',
 'centroid_displacement_m',
 'centroid_speed_m_per_h',
 'spread_bearing_deg',
 'spread_bearing_sin',
 'spread_bearing_cos',
 'dist_min_ci_0_5h',
 'dist_std_ci_0_5h',
 'dist_change_ci_0_5h',
 'dist_slope_ci_0_5h',
 'closing_speed_m_per_h',
 'closing_speed_abs_m_per_h',
 'projected_advance_m',
 'dist_accel_m_per_h2',
 'dist_fit_r2_0_5h',
 'alignment_cos',
 'alignment_abs',
 'cross_track_component',
 'along_track_speed',
 'event_start_hour',
 'event_start_dayofweek',
 'event_start_month']

In [10]:
X= trainDF[features].copy()
y_time  = trainDF[CFG.TIME_COL].values
y_event = trainDF[CFG.EVENT_COL].values

In [11]:
X_test = testDF[features].copy()
X_test

,event_id,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h,area_first_ha,area_growth_abs_0_5h,area_growth_rel_0_5h,area_growth_rate_ha_per_h,log1p_area_first,log1p_growth,...,projected_advance_m,dist_accel_m_per_h2,dist_fit_r2_0_5h,alignment_cos,alignment_abs,cross_track_component,along_track_speed,event_start_hour,event_start_dayofweek,event_start_month
0,10662602,1,0.000000,1,2.452217,0.000000,0.00000,0.000000,1.239017,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,3,7
1,13353600,1,0.000000,1,131.669588,0.000000,0.00000,0.000000,4.887862,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,22,0,8
2,13942327,1,0.000000,1,6.723104,0.000000,0.00000,0.000000,2.044216,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2,6,7
3,16112781,1,0.000000,1,285.416736,0.000000,0.00000,0.000000,5.657448,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,1,7
4,17132808,7,3.459331,0,61.098604,12.516633,0.20486,3.618224,4.128724,2.603921,...,13.54413,-22.687575,0.044572,0.158550,0.158550,-24.414806,3.920562,23,5,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,94627327,2,4.614923,0,24.438326,0.000000,0.00000,0.000000,3.236257,0.000000,...,0.00000,0.000000,0.000000,0.763809,0.763809,0.000000,0.000000,22,0,6
91,96570675,1,0.000000,1,155.843418,0.000000,0.00000,0.000000,5.055248,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,21,2,7
92,97225766,1,0.000000,1,87.761553,0.000000,0.00000,0.000000,4.485954,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4,1,7
93,98446281,2,4.335964,0,11.391108,0.000000,0.00000,0.000000,2.516979,0.000000,...,0.00000,0.000000,0.000000,-0.305683,0.305683,0.000000,-0.000000,17,0,9


In [12]:
len(y_time), len(y_event)

(221, 221)

## Basic preprocessing: scale numerical features

In [13]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

In [14]:
X_scaled , X_test_scaled

(array([[-1.71684052,  0.36402839,  1.89452231, ...,  0.45172677,
          0.58808168, -1.13676445],
        [-1.68227708, -0.02462028,  0.10959414, ..., -1.4462127 ,
          0.58808168, -0.49913769],
        [-1.67476526,  0.75267706,  2.18996854, ...,  0.83131466,
          0.58808168,  0.77611583],
        ...,
        [ 1.75722117, -0.02462028,  1.57474215, ...,  0.32519747,
         -0.93495798,  1.41374259],
        [ 1.80781574, -0.41326895, -0.56505413, ..., -0.05439042,
         -1.44263787,  0.77611583],
        [ 1.81853832, -0.41326895, -0.56505413, ...,  0.70478537,
         -0.93495798,  0.13848907]]),
 array([[-1.72602819, -0.41326895, -0.56505413, ..., -1.95232989,
          0.08040179,  0.13848907],
        [-1.61846471, -0.41326895, -0.56505413, ...,  0.83131466,
         -1.44263787,  0.77611583],
        [-1.59493235, -0.41326895, -0.56505413, ..., -1.6992713 ,
          1.60344145,  0.13848907],
        ...,
        [ 1.73403968, -0.41326895, -0.56505413, ..., -

## 2. Prepare XGBoost AFT labels
### Lower & upper bounds for interval-censored style (most common for AFT in XGBoost)


In [15]:
def prepare_aft_labels(time, event, max_time=72.1):
    """
    For right-censored data:
      - observed event:  [time, time]
      - censored       :  [time, +inf]  → represent +inf as large number
    XGBoost AFT expects two columns: label_lower, label_upper
    """
    lower = time.copy()
    upper = np.where(event == 1, time, max_time * 5)   # large value ≈ +∞
    return np.column_stack((lower, upper))


### 3. Create DMatrix (XGBoost native format)


### Found Bug
##### Many older blog posts / code snippets stack lower/upper into a single 2-column label=, but that's not how XGBoost officially handles AFT anymore (since ~1.2–1.6+ when AFT was properly integrated). The docs explicitly show separate label_lower_bound / label_upper_bound for interval-censored survival.

### new approach
The key fix (using label_lower_bound + label_upper_bound instead of a 2-column label=) resolved the "multioutput not supported" error, and the manual cross-validation loop is successfully training XGBoost AFT models on each fold without crashing.

In [48]:
# Found bug for combine lower, upper into a single label 
# y_aft = prepare_aft_labels(y_time, y_event)
# y_aft.shape
# dtrain = xgb.DMatrix(X_scaled, label=y_aft)
# dtrain


In [42]:

# new approach
max_censor = 72.1 * 5   # or 1000, whatever large finite value you used before

lower_bound = y_time.copy()                     # always the observed/censored time
upper_bound = np.where(y_event == 1, y_time, max_censor)   # finite for events, large for censored

# Create DMatrix correctly for AFT
dtrain = xgb.DMatrix(
    X_scaled,
    label_lower_bound=lower_bound,
    label_upper_bound=upper_bound
)

## 4. XGBoost AFT parameters
####    Choose distribution: 'normal', 'logistic', 'extreme' (logWeibull), 'gamma', etc.

In [49]:

params = {
    'objective': 'survival:aft',
    'aft_loss_distribution': 'normal',          # ← try 'logistic', 'extreme', 'gamma'
    'eval_metric': 'aft-nloglik',
    'learning_rate': 0.03,
    'max_depth': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    'min_child_weight': 3,
    'tree_method': 'hist',          # faster on Kaggle
    'seed': 42
}

In [50]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

print("Event rate:", y_event.mean())
print("Number of events:", y_event.sum(), "/", len(y_event))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Force creation of clean list-of-lists
folds_list = []
for i, (train_idx, val_idx) in enumerate(skf.split(X_scaled, y_event)):
    # Ensure we have numpy array → then convert to plain list
    val_list = val_idx.tolist() if isinstance(val_idx, np.ndarray) else list(val_idx)
    folds_list.append(val_list)
    
    print(f"Fold {i}: {len(val_list)} samples → indices type = {type(val_list)}, first 5 = {val_list[:5]}")

# Final safety checks
print("\nFinal checks:")
print("→ folds_list type:", type(folds_list))
print("→ number of folds:", len(folds_list))
print("→ all elements are lists?", all(isinstance(f, list) for f in folds_list))
print("→ lengths:", [len(f) for f in folds_list])
print("→ sample from fold 0:", folds_list[0][:8] if folds_list else "EMPTY!")

Event rate: 0.31221719457013575
Number of events: 69 / 221
Fold 0: 45 samples → indices type = <class 'list'>, first 5 = [10, 11, 13, 14, 15]
Fold 1: 44 samples → indices type = <class 'list'>, first 5 = [2, 9, 18, 38, 39]
Fold 2: 44 samples → indices type = <class 'list'>, first 5 = [0, 5, 6, 12, 23]
Fold 3: 44 samples → indices type = <class 'list'>, first 5 = [4, 17, 21, 24, 30]
Fold 4: 44 samples → indices type = <class 'list'>, first 5 = [1, 3, 7, 8, 19]

Final checks:
→ folds_list type: <class 'list'>
→ number of folds: 5
→ all elements are lists? True
→ lengths: [45, 44, 44, 44, 44]
→ sample from fold 0: [10, 11, 13, 14, 15, 16, 22, 25]


In [51]:


# ────────────────────────────────────────────────
# Reuse your existing folds_list (list of val indices)
# ────────────────────────────────────────────────

nfold = 5
best_ntree_list = []
aft_nloglik_list = []

skf = StratifiedKFold(n_splits=nfold, shuffle=True, random_state=42)
# but since you already have folds_list, we can use it

for fold_idx, val_indices in enumerate(folds_list):
    print(f"\nFold {fold_idx + 1}/{nfold}")
    
    train_mask = np.ones(len(X_scaled), dtype=bool)
    train_mask[val_indices] = False
    train_indices = np.where(train_mask)[0]
    
    dtrain_fold = dtrain.slice(train_indices.tolist())   # or train_indices
    dval_fold   = dtrain.slice(val_indices)
    
    watchlist = [(dtrain_fold, 'train'), (dval_fold, 'eval')]
    
    evals_result = {}
    model_fold = xgb.train(
        params,
        dtrain_fold,
        num_boost_round=2000,
        evals=watchlist,
        evals_result=evals_result,
        early_stopping_rounds=80,
        verbose_eval=50
    )
    
    best_ntree = model_fold.best_iteration + 1
    best_score = evals_result['eval']['aft-nloglik'][best_ntree - 1]
    
    best_ntree_list.append(best_ntree)
    aft_nloglik_list.append(best_score)
    
    print(f"Fold best iteration: {best_ntree}, aft-nloglik: {best_score:.6f}")

print("\nCV summary:")
print(f"Mean best ntree: {np.mean(best_ntree_list):.1f}")
print(f"Mean aft-nloglik: {np.mean(aft_nloglik_list):.6f} ± {np.std(aft_nloglik_list):.6f}")


Fold 1/5
[0]	train-aft-nloglik:9.95950	eval-aft-nloglik:9.71248
[50]	train-aft-nloglik:1.71894	eval-aft-nloglik:2.07515
[100]	train-aft-nloglik:1.06541	eval-aft-nloglik:1.59239
[150]	train-aft-nloglik:0.98522	eval-aft-nloglik:1.55379
[200]	train-aft-nloglik:0.96010	eval-aft-nloglik:1.55491
[239]	train-aft-nloglik:0.94946	eval-aft-nloglik:1.55823
Fold best iteration: 160, aft-nloglik: 1.549982

Fold 2/5
[0]	train-aft-nloglik:9.83461	eval-aft-nloglik:10.25478
[50]	train-aft-nloglik:1.70783	eval-aft-nloglik:2.36400
[100]	train-aft-nloglik:1.04252	eval-aft-nloglik:1.73652
[150]	train-aft-nloglik:0.94888	eval-aft-nloglik:1.68393
[200]	train-aft-nloglik:0.91481	eval-aft-nloglik:1.67191
[250]	train-aft-nloglik:0.89620	eval-aft-nloglik:1.67805
[300]	train-aft-nloglik:0.88531	eval-aft-nloglik:1.67992
[314]	train-aft-nloglik:0.88304	eval-aft-nloglik:1.68126
Fold best iteration: 235, aft-nloglik: 1.671218

Fold 3/5
[0]	train-aft-nloglik:9.62050	eval-aft-nloglik:11.14619
[50]	train-aft-nloglik:1.

In [57]:
# Final model on full dataset
watchlist_full = [(dtrain, 'train')]   # no separate validation here

evals_result_full = {}
final_model = xgb.train(
    params,
    dtrain,
    num_boost_round=300,                   # a bit higher than CV average
    evals=watchlist_full,
    evals_result=evals_result_full,
    early_stopping_rounds=80,
    verbose_eval=20
)

best_ntree_final = final_model.best_iteration + 1
print(f"Final model stopped at iteration: {best_ntree_final}")

[0]	train-aft-nloglik:9.91011
[20]	train-aft-nloglik:4.12866
[40]	train-aft-nloglik:2.16061
[60]	train-aft-nloglik:1.45285
[80]	train-aft-nloglik:1.17966
[100]	train-aft-nloglik:1.06481
[120]	train-aft-nloglik:1.00873
[140]	train-aft-nloglik:0.97995
[160]	train-aft-nloglik:0.96208
[180]	train-aft-nloglik:0.94915
[200]	train-aft-nloglik:0.94002
[220]	train-aft-nloglik:0.93261
[240]	train-aft-nloglik:0.92743
[260]	train-aft-nloglik:0.92267
[280]	train-aft-nloglik:0.91826
[299]	train-aft-nloglik:0.91539
Final model stopped at iteration: 300


In [58]:
from scipy.stats import lognorm
import numpy as np

TIME_HORIZONS = [12, 24, 48, 72]

def compute_oof_probs(models, dtrain, sigma):
    oof_probs = np.zeros((len(X_scaled), len(TIME_HORIZONS)))
    for fold_idx, val_indices in enumerate(folds_list):
        dval = dtrain.slice(val_indices)
        mu_log = models[fold_idx].predict(dval)   # you need to save the 5 models!
        # For 'normal' → log(T) ~ N(mu_log, sigma)
        surv = 1 - lognorm.cdf(TIME_HORIZONS, s=sigma, scale=np.exp(mu_log))
        oof_probs[val_indices] = surv.T   # shape: (n_val, n_horizons)
    return oof_probs

#### Note:
- You need to collect the 5 fold models first (small modification to loop)
- Then try sigmas = [0.5, 0.7, 0.9, 1.0, 1.2, 1.5]
- Pick sigma that minimizes average (1 - S(t)) Brier score on oof vs true event/censoring

In [68]:
dtest = xgb.DMatrix(X_test_scaled)
mu_log_test = final_model.predict(dtest)

# Example with sigma=1.0 (tune this!)
sigma = 1.0
surv_probs = 1 - lognorm.cdf(
    np.array(TIME_HORIZONS),
    s=sigma,
    scale=np.exp(mu_log_test)[:, None]
)

submission = pd.DataFrame({
    'event_id': testDF['event_id'],
    'prob_12h': 1 - surv_probs[:, 0],
    'prob_24h': 1 - surv_probs[:, 1],
    'prob_48h': 1 - surv_probs[:, 2],
    'prob_72h': 1 - surv_probs[:, 3]
})

submission.to_csv("submission.csv", index=False)

/tmp/ipykernel_55/5175036.py:9: RuntimeWarning: overflow encountered in exp
  scale=np.exp(mu_log_test)[:, None]
